# Layer 1: Controlled Intake Engine (Colab Runner)

Notebook này chạy **Layer 1: Controlled Intake Engine** (`controlled_intake_engine.py`) trên Google Colab hoặc môi trường local để tiếp nhận 4 file đầu vào từ Layer 0:

1. `service_case_feedback.csv`
2. `curated_correspondence.csv`
3. `survey_responses.csv`
4. `batch_manifest.json`

### Outputs (`intake_output/`)
- `immutable_original_store/`: Lưu nguyên bản `original_*.csv` (`original_text`).
- `normalized_store/`: Lưu `normalized_*.csv` (đã che PII `redacted_text`, gắn `priority_flag`), `batch_manifest_ledger.csv`, `coverage_report.json`, `intake_state.json`, và `invalidation_events.jsonl`.
- `quarantine_queue/`: Lưu `quarantine_*.csv` (lỗi schema, vượt 8.000 ký tự `unanalyzed`, hoặc bản ghi trùng lặp khi retry).

## Module 0: Environment & Folder Setup (Google Drive)

Đặt thư mục gốc trên Google Drive theo cấu trúc cây nhỏ sau để notebook tự động tìm thấy code và 4 file đầu vào:

```text
MyDrive/__SongNhat/5_layer_core/           <-- PROJECT_ROOT (chỉnh 1 dòng dưới nếu khác)
├── 0_ready_for_intake/
│   └── ready_for_intake_4_files/
│       ├── service_case_feedback.csv
│       ├── curated_correspondence.csv
│       ├── survey_responses.csv
│       └── batch_manifest.json
└── 1_controlled_intake/
    ├── 1_controlled_intake.ipynb
    ├── controlled_intake_engine.py
    ├── test_controlled_intake.py
    └── intake_output/                     <-- tự động tạo khi chạy
```

In [ ]:
import json
import shutil
import sys
from pathlib import Path

import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/__SongNhat/5_layer_core")
else:
    PROJECT_ROOT = Path.cwd().parent

WORK_DIR = PROJECT_ROOT / "1_controlled_intake"
INPUT_DIR = PROJECT_ROOT / "0_ready_for_intake" / "ready_for_intake_4_files"
OUTPUT_DIR = WORK_DIR / "intake_output"

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"WORK_DIR     : {WORK_DIR}")
print(f"INPUT_DIR    : {INPUT_DIR}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")

## Module 1: Verify Code & Input Data

Kiểm tra `controlled_intake_engine.py` trong `WORK_DIR` và 4 file Layer 0 trong `INPUT_DIR`.

In [ ]:
REQUIRED_INPUTS = [
    "service_case_feedback.csv",
    "curated_correspondence.csv",
    "survey_responses.csv",
    "batch_manifest.json",
]

engine_path = WORK_DIR / "controlled_intake_engine.py"
assert engine_path.exists(), f"Không tìm thấy {engine_path}. Hãy kiểm tra lại PROJECT_ROOT."
print(f"OK: {engine_path.name} ({engine_path.stat().st_size:,} bytes)")

for fname in REQUIRED_INPUTS:
    fpath = INPUT_DIR / fname
    assert fpath.exists(), f"Không tìm thấy {fpath}. Hãy kiểm tra lại cấu trúc folder."
    print(f"OK: {fname} ({fpath.stat().st_size:,} bytes)")

## Module 2: Run Controlled Intake Engine

Khởi tạo `ControlledIntakeEngine` và xử lý batch từ `INPUT_DIR` vào `OUTPUT_DIR`.

In [ ]:
from controlled_intake_engine import ControlledIntakeEngine

engine = ControlledIntakeEngine(output_dir=OUTPUT_DIR)
coverage_report = engine.process_batch(INPUT_DIR)

print("=== BATCH SUMMARY ===")
print(f"batch_id                  : {coverage_report['batch_id']}")
print(f"source_version            : {coverage_report['source_version']}")
print(f"load_mode                 : {coverage_report['load_mode']}")
print(f"is_stale                  : {coverage_report['is_stale']}")
print(f"deletion_tombstones       : {coverage_report['deletion_tombstones_processed']}")
print(f"invalidation_events       : {coverage_report['invalidation_events_emitted']}")

reconciliation_df = pd.DataFrame(coverage_report["reconciliation"])
display(reconciliation_df)

## Module 3: Inspect Outputs (Normalized, Quarantine, Invalidation)

Kiểm tra nhanh dữ liệu đầu ra sau khi chạy Layer 1.

In [ ]:
normalized_dir = OUTPUT_DIR / "normalized_store"
quarantine_dir = OUTPUT_DIR / "quarantine_queue"

print("--- Sample: normalized_curated_correspondence.csv (PII redacted + priority_flag) ---")
corr_df = pd.read_csv(normalized_dir / "normalized_curated_correspondence.csv")
cols_to_show = [c for c in ["record_id", "author_role", "audience", "priority_flag", "priority_keywords", "text"] if c in corr_df.columns]
display(corr_df[cols_to_show].head(5))

print("\n--- Quarantine Files ---")
for qfile in sorted(quarantine_dir.glob("*.csv")):
    qdf = pd.read_csv(qfile)
    print(f"{qfile.name}: {len(qdf)} rows")
    if "quarantine_reason" in qdf.columns:
        print(qdf["quarantine_reason"].value_counts().to_dict())

inv_path = normalized_dir / "invalidation_events.jsonl"
if inv_path.exists():
    print("\n--- Invalidation Events (DAT 08) ---")
    for line in inv_path.read_text(encoding="utf-8").splitlines()[:5]:
        print(line)

## Module 4: Run Acceptance Unit Tests

Chạy `test_controlled_intake.py` để xác nhận các kiểm thử `DAT 01` - `DAT 08`, `SEC 03`.

In [ ]:
import unittest

test_file = WORK_DIR / "test_controlled_intake.py"
if test_file.exists():
    suite = unittest.defaultTestLoader.discover(start_dir=str(WORK_DIR), pattern="test_controlled_intake.py")
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    assert result.wasSuccessful(), "Some acceptance tests failed!"
else:
    print("Skip: test_controlled_intake.py not found in WORK_DIR.")

## Module 5: Package & Download Output (Colab)

Nén toàn bộ `intake_output/` thành `intake_output.zip` để tải về máy hoặc chuyển tiếp sang Layer 2.

In [ ]:
if IN_COLAB:
    print(f"Output đã được lưu trực tiếp lên Google Drive tại: {OUTPUT_DIR}")
else:
    zip_base = WORK_DIR / "intake_output"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(OUTPUT_DIR)))
    print(f"Created archive: {zip_path} ({zip_path.stat().st_size:,} bytes)")